In [9]:
from pathlib import Path

import pandas as pd

RAW_DATA_PATH = Path("Teen_Mental_Health_Dataset.csv")
OUTPUT_DIR = Path("processed")
OUTPUT_DIR.mkdir(exist_ok=True)

EXPECTED_COLUMNS = [
    "age",
    "gender",
    "daily_social_media_hours",
    "platform_usage",
    "sleep_hours",
    "screen_time_before_sleep",
    "academic_performance",
    "physical_activity",
    "social_interaction_level",
    "stress_level",
    "anxiety_level",
    "addiction_level",
    "depression_label",
]

RAW_DATA_PATH.resolve(), OUTPUT_DIR.resolve()

(PosixPath('/Users/thanhle/thanhle_drive/JRM/ Final_Project/Teen_Mental_Health_Dataset.csv'),
 PosixPath('/Users/thanhle/thanhle_drive/JRM/ Final_Project/processed'))

In [10]:
df_raw = pd.read_csv(RAW_DATA_PATH)

missing_columns = sorted(set(EXPECTED_COLUMNS) - set(df_raw.columns))
extra_columns = sorted(set(df_raw.columns) - set(EXPECTED_COLUMNS))

if missing_columns:
    raise ValueError(f"Missing expected columns: {missing_columns}")

print(f"Rows: {len(df_raw):,}")
print(f"Columns: {len(df_raw.columns):,}")
if extra_columns:
    print(f"Extra columns ignored: {extra_columns}")

df_raw.head()

Rows: 1,200
Columns: 13


,age,gender,daily_social_media_hours,platform_usage,sleep_hours,screen_time_before_sleep,academic_performance,physical_activity,social_interaction_level,stress_level,anxiety_level,addiction_level,depression_label
0,14,male,7.9,Instagram,7.4,2.9,3.01,1.5,low,2,2,1,0
1,19,female,1.9,TikTok,8.0,2.9,3.22,0.8,high,8,1,10,0
2,17,female,1.3,Instagram,7.6,0.5,3.92,0.0,high,2,4,2,0
3,15,male,7.4,TikTok,6.9,1.6,3.48,0.8,medium,1,7,9,0
4,15,female,4.7,Both,4.9,3.0,2.37,1.4,medium,3,5,2,0


In [11]:
df = df_raw[EXPECTED_COLUMNS].copy()

text_columns = ["gender", "platform_usage", "social_interaction_level"]
for column in text_columns:
    df[column] = df[column].astype("string").str.strip()

category_maps = {
    "gender": {
        "male": "Male",
        "female": "Female",
    },
    "platform_usage": {
        "instagram": "Instagram",
        "tiktok": "TikTok",
        "both": "Both",
    },
    "social_interaction_level": {
        "low": "Low",
        "medium": "Medium",
        "high": "High",
    },
}

for column, mapping in category_maps.items():
    df[column] = df[column].str.lower().map(mapping)
    if df[column].isna().any():
        raise ValueError(f"Unknown values in {column}: {df_raw.loc[df[column].isna(), column].unique().tolist()}")

numeric_columns = [
    "age",
    "daily_social_media_hours",
    "sleep_hours",
    "screen_time_before_sleep",
    "academic_performance",
    "physical_activity",
    "stress_level",
    "anxiety_level",
    "addiction_level",
    "depression_label",
]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="raise")

# Match the report schema: 13-15 Early Teen, 16-17 Middle Teen, 18-19 Late Teen.
age_bins = [12, 15, 17, 19]
age_labels = ["Early Teen", "Middle Teen", "Late Teen"]
df["age_group"] = pd.cut(df["age"], bins=age_bins, labels=age_labels, right=True)

if df["age_group"].isna().any():
    raise ValueError(f"Ages outside configured teen bands: {sorted(df.loc[df['age_group'].isna(), 'age'].unique())}")

df.isna().sum()

age                         0
gender                      0
daily_social_media_hours    0
platform_usage              0
sleep_hours                 0
screen_time_before_sleep    0
academic_performance        0
physical_activity           0
social_interaction_level    0
stress_level                0
anxiety_level               0
addiction_level             0
depression_label            0
age_group                   0
dtype: int64

In [12]:
dim_age = (
    df[["age", "age_group"]]
    .drop_duplicates()
    .sort_values("age")
    .reset_index(drop=True)
    .rename(columns={"age": "Age", "age_group": "AgeGroup"})
)
dim_age.insert(0, "AgeKey", range(1, len(dim_age) + 1))


def build_ordered_dimension(
    values: list[str],
    key_name: str,
    value_name: str,
    observed: pd.Series,
) -> pd.DataFrame:
    missing_values = sorted(set(observed.dropna()) - set(values))
    if missing_values:
        raise ValueError(f"Unexpected {value_name} values: {missing_values}")

    dimension = pd.DataFrame({value_name: values})
    dimension.insert(0, key_name, range(1, len(dimension) + 1))
    return dimension


dim_gender = build_ordered_dimension(
    ["Male", "Female"],
    "GenderKey",
    "Gender",
    df["gender"],
)
dim_platform = build_ordered_dimension(
    ["Instagram", "TikTok", "Both"],
    "PlatformKey",
    "PlatformUsage",
    df["platform_usage"],
)

dim_social_interaction = build_ordered_dimension(
    ["Low", "Medium", "High"],
    "SocialInteractionKey",
    "SocialInteractionLevel",
    df["social_interaction_level"],
)
dim_social_interaction["InteractionRank"] = dim_social_interaction["SocialInteractionKey"]

dim_age, dim_gender, dim_platform, dim_social_interaction

(   AgeKey  Age     AgeGroup
 0       1   13   Early Teen
 1       2   14   Early Teen
 2       3   15   Early Teen
 3       4   16  Middle Teen
 4       5   17  Middle Teen
 5       6   18    Late Teen
 6       7   19    Late Teen,
    GenderKey  Gender
 0          1    Male
 1          2  Female,
    PlatformKey PlatformUsage
 0            1     Instagram
 1            2        TikTok
 2            3          Both,
    SocialInteractionKey SocialInteractionLevel  InteractionRank
 0                     1                    Low                1
 1                     2                 Medium                2
 2                     3                   High                3)

In [13]:
fact = (
    df.reset_index(names="RespondentID")
    .assign(RespondentID=lambda data: data["RespondentID"] + 1)
    .merge(dim_age, left_on=["age", "age_group"], right_on=["Age", "AgeGroup"], how="left")
    .merge(dim_gender, left_on="gender", right_on="Gender", how="left")
    .merge(dim_platform, left_on="platform_usage", right_on="PlatformUsage", how="left")
    .merge(
        dim_social_interaction,
        left_on="social_interaction_level",
        right_on="SocialInteractionLevel",
        how="left",
    )
)

fact_teen_mental_health = fact[
    [
        "RespondentID",
        "AgeKey",
        "GenderKey",
        "PlatformKey",
        "SocialInteractionKey",
        "daily_social_media_hours",
        "sleep_hours",
        "screen_time_before_sleep",
        "academic_performance",
        "physical_activity",
        "stress_level",
        "anxiety_level",
        "addiction_level",
        "depression_label",
    ]
].rename(
    columns={
        "daily_social_media_hours": "DailySocialMediaHours",
        "sleep_hours": "SleepHours",
        "screen_time_before_sleep": "ScreenTimeBeforeSleep",
        "academic_performance": "AcademicPerformance",
        "physical_activity": "PhysicalActivity",
        "stress_level": "StressLevel",
        "anxiety_level": "AnxietyLevel",
        "addiction_level": "AddictionLevel",
        "depression_label": "DepressionLabel",
    }
)

fact_teen_mental_health.head()

,RespondentID,AgeKey,GenderKey,PlatformKey,SocialInteractionKey,DailySocialMediaHours,SleepHours,ScreenTimeBeforeSleep,AcademicPerformance,PhysicalActivity,StressLevel,AnxietyLevel,AddictionLevel,DepressionLabel
0,1,2,1,1,1,7.9,7.4,2.9,3.01,1.5,2,2,1,0
1,2,7,2,2,3,1.9,8.0,2.9,3.22,0.8,8,1,10,0
2,3,5,2,1,3,1.3,7.6,0.5,3.92,0.0,2,4,2,0
3,4,3,1,2,2,7.4,6.9,1.6,3.48,0.8,1,7,9,0
4,5,3,2,3,2,4.7,4.9,3.0,2.37,1.4,3,5,2,0


In [14]:
dimension_tables = {
    "Dim_Age": dim_age,
    "Dim_Gender": dim_gender,
    "Dim_Platform": dim_platform,
    "Dim_SocialInteraction": dim_social_interaction,
}

for table_name, table in dimension_tables.items():
    key_column = table.columns[0]
    if table[key_column].duplicated().any():
        raise ValueError(f"Duplicate keys found in {table_name}.{key_column}")

required_fact_keys = ["AgeKey", "GenderKey", "PlatformKey", "SocialInteractionKey"]
missing_fact_keys = fact_teen_mental_health[required_fact_keys].isna().sum()
if missing_fact_keys.any():
    raise ValueError(f"Missing foreign keys:\n{missing_fact_keys[missing_fact_keys > 0]}")

if fact_teen_mental_health["RespondentID"].duplicated().any():
    raise ValueError("RespondentID must be unique in Fact_TeenMentalHealth")

print("Validation passed")
print(f"Fact rows: {len(fact_teen_mental_health):,}")
print("Dimension rows:")
for table_name, table in dimension_tables.items():
    print(f"- {table_name}: {len(table):,}")

Validation passed
Fact rows: 1,200
Dimension rows:
- Dim_Age: 7
- Dim_Gender: 2
- Dim_Platform: 3
- Dim_SocialInteraction: 3


In [15]:
exports = {
    "Dim_Age.csv": dim_age,
    "Dim_Gender.csv": dim_gender,
    "Dim_Platform.csv": dim_platform,
    "Dim_SocialInteraction.csv": dim_social_interaction,
    "Fact_TeenMentalHealth.csv": fact_teen_mental_health,
}

for file_name, table in exports.items():
    table.to_csv(OUTPUT_DIR / file_name, index=False)

print("Exported files:")
for file_name in exports:
    print(f"- {OUTPUT_DIR / file_name}")

Exported files:
- processed/Dim_Age.csv
- processed/Dim_Gender.csv
- processed/Dim_Platform.csv
- processed/Dim_SocialInteraction.csv
- processed/Fact_TeenMentalHealth.csv


In [16]:
for table_name, table in {
    **dimension_tables,
    "Fact_TeenMentalHealth": fact_teen_mental_health,
}.items():
    display(table.head())

,AgeKey,Age,AgeGroup
0,1,13,Early Teen
1,2,14,Early Teen
2,3,15,Early Teen
3,4,16,Middle Teen
4,5,17,Middle Teen


,GenderKey,Gender
0,1,Male
1,2,Female


,PlatformKey,PlatformUsage
0,1,Instagram
1,2,TikTok
2,3,Both


,SocialInteractionKey,SocialInteractionLevel,InteractionRank
0,1,Low,1
1,2,Medium,2
2,3,High,3


,RespondentID,AgeKey,GenderKey,PlatformKey,SocialInteractionKey,DailySocialMediaHours,SleepHours,ScreenTimeBeforeSleep,AcademicPerformance,PhysicalActivity,StressLevel,AnxietyLevel,AddictionLevel,DepressionLabel
0,1,2,1,1,1,7.9,7.4,2.9,3.01,1.5,2,2,1,0
1,2,7,2,2,3,1.9,8.0,2.9,3.22,0.8,8,1,10,0
2,3,5,2,1,3,1.3,7.6,0.5,3.92,0.0,2,4,2,0
3,4,3,1,2,2,7.4,6.9,1.6,3.48,0.8,1,7,9,0
4,5,3,2,3,2,4.7,4.9,3.0,2.37,1.4,3,5,2,0
